In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/main_signal_filtered/"

In [26]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*02:01']

# 2. Train HMMS

In [27]:
from hmm_logic_methods import train_model, save_model


NUM_RUNS = 2
TARGET_LENGTH = 9
TARGET_LENGTHS= [9]

## 2.1 Filter data

In [28]:
for allele_name in ALLELES:
    for split_num in per_allele_per_kfold_per_length_binders_train[allele_name].keys():
        for target_length in per_allele_per_kfold_per_length_binders_train[allele_name][split_num].keys():
            if target_length == 9:
                data_array = per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length]
                per_allele_per_kfold_per_length_binders_train[allele_name][split_num][target_length] = np.array([s for s in data_array if s[1] not in ['V','L']])

                data_array = per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length]
                per_allele_per_kfold_per_length_binders_test[allele_name][split_num][target_length] = np.array([s for s in data_array if s[1]  not in ['V','L']])

## 2.2 Training process

In [29]:
cutoff = 0.00005
train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][9])
## Stop threshold Hardcoded here
stop_threshold =  abs(cutoff * train_size * -22)
stop_threshold

10.1662

In [30]:
%%time
TARGET_LENGTHS = [9]
algorithm = 'baum-welch'

per_allele_models = dict()
per_name_models = dict()
per_allele_histories = dict()
for run_num in range(NUM_RUNS):
    print(f"run {run_num}")
    for allele_name in ALLELES:
        print()
        per_kfold_per_length_data =  per_allele_per_kfold_per_length_binders_train[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        verbose=True,
                                        multiple_check_input=False,
                                        algorithm=algorithm,
                                        name_prefix=f"{run_num}_run",
                                        num_states=18,
                                        n_jobs=10,
                                        stop_threshold=stop_threshold
                                        )
        per_allele_models[allele_name] = models
        per_allele_histories[allele_name] = histories
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in per_allele_models[allele_name].items():
            history = per_allele_histories[allele_name][split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


run 0

split 0
train size 9242
Shuffling for fairness 
Simple training: Train Length 9242, init_batch_size None
[1] Improvement: 13322.45023393957	Time (s): 1.662
[2] Improvement: 471.4803304388188	Time (s): 1.713
[3] Improvement: 675.196295249043	Time (s): 1.769
[4] Improvement: 952.014964012662	Time (s): 1.757
[5] Improvement: 1144.292132280767	Time (s): 1.778
[6] Improvement: 1046.9187047854357	Time (s): 1.682
[7] Improvement: 735.6621950968693	Time (s): 1.677
[8] Improvement: 488.48244898900157	Time (s): 1.988
[9] Improvement: 379.9571957082662	Time (s): 1.968
[10] Improvement: 342.5355380366964	Time (s): 2.086
[11] Improvement: 330.2951890800905	Time (s): 1.808
[12] Improvement: 329.9943657890253	Time (s): 1.728
[13] Improvement: 338.9609173206263	Time (s): 1.677
[14] Improvement: 357.74683555640513	Time (s): 1.688
[15] Improvement: 389.7419099104882	Time (s): 1.752
[16] Improvement: 440.32480775614385	Time (s): 1.746
[17] Improvement: 515.7511306538945	Time (s): 1.766
[18] Improv

In [31]:
from hmm_visualization_methods import *

In [32]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 280 ms


In [33]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 25.1 s


In [34]:
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-0) (Delete zero edges)
Converting graph for model (1_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-0) (Delete zero edges)


In [24]:
model.in_transitions()

AttributeError: 'pomegranate.hmm.HiddenMarkovModel' object has no attribute 'in_transitions'

In [36]:
data_array = data_array[:10]

In [41]:
keymap = {'H': 0,
          'R': 1,
          'V': 2,
          'S': 3,
          'N': 4,
          'Q': 5,
          'W': 6,
          'K': 7,
          'M': 8,
          'D': 9,
          'I': 10,
          'Y': 11,
          'C': 12,
          'L': 13,
          'G': 14,
          'F': 15,
          'A': 16,
          'P': 17,
          'T': 18,
          'E': 19}

In [42]:
sample_X = np.array([[keymap[char] for char in item] for item in data_array], dtype=object)

In [43]:
sample_X

array([[13, 13, 13, 19, 2, 3, 19, 19, 13],
       [2, 2, 2, 7, 10, 13, 7, 17, 2],
       [16, 2, 8, 5, 19, 2, 17, 13, 16],
       [10, 13, 13, 3, 5, 10, 2, 15, 10],
       [15, 18, 10, 15, 1, 18, 10, 3, 2],
       [10, 13, 9, 14, 19, 4, 12, 18, 13],
       [16, 10, 9, 15, 13, 13, 5, 1, 6],
       [13, 13, 10, 16, 10, 13, 14, 17, 13],
       [16, 16, 15, 9, 14, 14, 15, 18, 2],
       [0, 13, 11, 16, 1, 5, 13, 18, 13]], dtype=object)

In [44]:
from pomegranate import MarkovNetwork

In [45]:
model = MarkovNetwork.from_samples(sample_X)

KeyboardInterrupt: 